# Notebook 3 — Model Training (Distributed ML with MLlib)

**Module:** 7006SCN Machine Learning and Big Data — Coventry University  
**Project:** Scalable Fake News Detection Using Distributed ML on Common Crawl  

---

## Models trained
| # | Model | MLlib class | Why included |
|---|-------|------------|---------------|
| 1 | Logistic Regression | `LogisticRegression` | Strong baseline; outputs probabilities for ROC |
| 2 | Linear SVC | `LinearSVC` | Margin-based; often best on high-dim text |
| 3 | Random Forest | `RandomForestClassifier` | Non-linear; provides feature importance |

## Distributed vs scikit-learn
- MLlib models partition data across executors; each executor trains on its shard.
- Gradient updates (LR, SVC) are aggregated via **AllReduce** across the cluster.
- Random Forest builds trees in parallel — Spark assigns *different trees* to different executors.
- scikit-learn is single-node: training time grows linearly with data size, bounded by local RAM.
- MLlib scales horizontally: add executors → near-linear speedup (see Notebook 4 scaling experiments).

In [ ]:
# ── 3.1  Bootstrap ──────────────────────────────────────────────────────
import os, sys, time
os.environ["JAVA_HOME"]             = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.16.8-hotspot"
os.environ["HADOOP_HOME"]           = r"C:\hadoop"
os.environ["PATH"]                  = r"C:\hadoop\bin;" + os.environ.get("PATH", "")
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel

spark = (
    SparkSession.builder
    .appName("FakeNewsDetection")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} | UI: {spark.sparkContext.uiWebUrl}")

In [ ]:
# ── 3.2  Load feature-engineered splits ─────────────────────────────────
FEATURES_BASE = "../data/parquet/features"

train_df = spark.read.parquet(f"{FEATURES_BASE}/train").persist(StorageLevel.MEMORY_AND_DISK)
val_df   = spark.read.parquet(f"{FEATURES_BASE}/val").persist(StorageLevel.MEMORY_AND_DISK)
test_df  = spark.read.parquet(f"{FEATURES_BASE}/test").persist(StorageLevel.MEMORY_AND_DISK)

print(f"Train: {train_df.count():>8,}  |  Val: {val_df.count():>6,}  |  Test: {test_df.count():>6,}")

In [ ]:
# ── 3.3  Model definitions + hyperparameter grids ───────────────────────
from pyspark.ml.classification import (
    LogisticRegression, LinearSVC, RandomForestClassifier
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator, MulticlassClassificationEvaluator
)

# ── Logistic Regression ── (small grid to keep CV tractable)
lr = LogisticRegression(
    featuresCol="features", labelCol="label",
    maxIter=100, family="binomial"
)
lr_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5])  # L2 vs elastic net
    .build()
)

# ── Linear SVC ──
svc = LinearSVC(featuresCol="features", labelCol="label", maxIter=100)
svc_grid = (
    ParamGridBuilder()
    .addGrid(svc.regParam, [0.01, 0.1])
    .build()
)

# ── Random Forest ── (direct fit — no CV — to avoid OOM with 2^14 features)
rf = RandomForestClassifier(
    featuresCol="features", labelCol="label",
    numTrees=100, maxDepth=5, maxBins=32, seed=42
)

print(f"LR grid size : {len(lr_grid)} combos × 3 folds = {len(lr_grid)*3} fits")
print(f"SVC grid size: {len(svc_grid)} combos × 3 folds = {len(svc_grid)*3} fits")
print(f"RF           : direct fit (no CV — prevents OOM on Windows)")

In [ ]:
# ── 3.4  Train LR & SVC with 3-fold CrossValidator ──────────────────────
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)

cv_results = {}
NUM_FOLDS = 3

for name, (estimator, grid) in [("LogisticRegression", (lr, lr_grid)),
                                  ("LinearSVC", (svc, svc_grid))]:
    print(f"\n{'='*60}")
    print(f"Training: {name}  |  Grid combos: {len(grid)}  |  Folds: {NUM_FOLDS}")
    print(f"{'='*60}")
    
    cv = CrossValidator(
        estimator=estimator,
        estimatorParamMaps=grid,
        evaluator=auc_evaluator,
        numFolds=NUM_FOLDS,
        parallelism=2, seed=42
    )
    
    t0 = time.time()
    cv_model = cv.fit(train_df)
    train_time = time.time() - t0
    
    best_model = cv_model.bestModel
    best_auc   = max(cv_model.avgMetrics)
    
    cv_results[name] = {
        "best_model": best_model,
        "best_auc":   best_auc,
        "train_time": train_time,
    }
    
    print(f"  Best CV AUC: {best_auc:.4f}  |  Time: {train_time:.1f}s")

# ── Train Random Forest directly (no CV) ──
print(f"\n{'='*60}")
print(f"Training: RandomForest (direct fit — numTrees=100, maxDepth=5)")
print(f"{'='*60}")

t0 = time.time()
rf_model = rf.fit(train_df)
rf_time = time.time() - t0

# Evaluate on validation
rf_preds = rf_model.transform(val_df)
rf_auc   = auc_evaluator.evaluate(rf_preds)

cv_results["RandomForest"] = {
    "best_model": rf_model,
    "best_auc":   rf_auc,
    "train_time": rf_time,
}
print(f"  Val AUC: {rf_auc:.4f}  |  Time: {rf_time:.1f}s")

In [ ]:
# ── 3.5  Evaluate all models on VALIDATION set ─────────────────────────
acc_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)

val_metrics = []
for name, res in cv_results.items():
    preds = res["best_model"].transform(val_df)
    accuracy = acc_eval.evaluate(preds)
    f1       = f1_eval.evaluate(preds)
    try:
        auc_v = auc_evaluator.evaluate(preds)
    except Exception:
        auc_v = None
    
    auc_str = f"{auc_v:.4f}" if auc_v is not None else "N/A"
    val_metrics.append({
        "model": name,
        "accuracy": round(accuracy, 4),
        "f1": round(f1, 4),
        "auc": round(auc_v, 4) if auc_v else "N/A",
        "train_time_s": round(res["train_time"], 1),
    })
    print(f"{name:25s}  Acc={accuracy:.4f}  F1={f1:.4f}  AUC={auc_str}")

import pandas as pd
val_metrics_df = pd.DataFrame(val_metrics)
print("\n", val_metrics_df.to_string(index=False))

In [ ]:
# ── 3.6  Save models — MLlib format ─────────────────────────────────────
from pathlib import Path

MODELS_DIR = Path("../data/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

for name, res in cv_results.items():
    mllib_path = str(MODELS_DIR / f"{name}_mllib")
    res["best_model"].write().overwrite().save(mllib_path)
    print(f"✓ {name} → {mllib_path}")

print("\nAll models saved.")

In [ ]:
# ── 3.7  Export model comparison for Tableau ────────────────────────────
val_metrics_df.to_csv("../tableau/model_comparison.csv", index=False)
print("✓ Exported model_comparison.csv for Tableau dashboard")
val_metrics_df

## 3.8  Model Stability Explanation

**How do we know the models are stable?**

1. **5-fold Cross-Validation:**  
   The `avgMetrics` from `CrossValidator` gives the mean AUC across 5 folds.
   Low variance across folds (check `cv_results[model]['avg_metrics']`) indicates
   the model is not overfitting to a particular data split.

2. **Train vs Validation gap:**  
   If train accuracy >> validation accuracy, the model overfits.
   Our LR with L1/L2 regularisation and RF with limited depth reduce this.

3. **Stratified splits:**  
   `sampleBy` preserves label ratios → models don't get biased toward
   the majority class in any fold.

4. **Final TEST set evaluation:**  
   The test set is held out from ALL training and hyperparameter selection.
   Results in Notebook 4 give the unbiased performance estimate.

In [ ]:
# ── Cleanup ──
train_df.unpersist()
val_df.unpersist()
test_df.unpersist()
# spark.stop()